# 高级界面功能

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [ ]:
!pip install datasets evaluate transformers[sentencepiece]
!pip install gradio

In [4]:
import random

import gradio as gr


def chat(message, history):
    # history 由 gr.State() 组件跨轮次传入，初次调用时为 None，用 or [] 初始化为空列表
    # Gradio 4.x 中消息格式从元组 (user, bot) 改为字典列表：
    #   {"role": "user", "content": "..."} 和 {"role": "assistant", "content": "..."}
    history = history or []

    # 根据用户消息前缀生成不同的回复（模拟简单规则对话）
    if message.startswith("How many"):
        response = random.randint(1, 10)
    elif message.startswith("How"):
        response = random.choice(["Great", "Good", "Okay", "Bad"])
    elif message.startswith("Where"):
        response = random.choice(["Here", "There", "Somewhere"])
    else:
        response = "I don't know"

    # 用新格式追加本轮对话：先追加用户消息，再追加机器人回复
    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": str(response)})

    # 同时返回给 gr.Chatbot()（显示）和 gr.State()（记忆）
    return history, history


iface = gr.Interface(
    chat,
    inputs=[gr.Textbox(label="Your message"), gr.State()],
    # type="messages" 告诉 Chatbot 使用新的字典消息格式
    outputs=[gr.Chatbot(), gr.State()],
    flagging_mode="never",
)
iface.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [ ]:
import requests
import tensorflow as tf

import gradio as gr

inception_net = tf.keras.applications.MobileNetV2()  # load the model

# Download human-readable labels for ImageNet.
response = requests.get("https://git.io/JJkYN")
labels = response.text.split("\n")


def classify_image(inp):
    inp = inp.reshape((-1, 224, 224, 3))
    inp = tf.keras.applications.mobilenet_v2.preprocess_input(inp)
    prediction = inception_net.predict(inp).flatten()
    return {labels[i]: float(prediction[i]) for i in range(1000)}


image = gr.Image(shape=(224, 224))
label = gr.Label(num_top_classes=3)

title = "Gradio Image Classifiction + Interpretation Example"
gr.Interface(
    fn=classify_image, inputs=image, outputs=label, interpretation="default", title=title
).launch()